---

# Dudley Economic Activity — Data Cleaning

---

## Objectives
- Inspect the dataset structure and geographical coverage.
- Check for missing values, duplicates and incorrect data types.
- Prepare the economic activity data for analysis.
- Save a cleaned copy while preserving the raw dataset.

## Inputs
- `raw_datasets/dudley_economic_activity.csv`

## Outputs

- Cleaned dataset: `cleaned_datasets/dudley_economic_activity_cleaned.csv`
- Documented data quality checks and cleaning decisions.

## Additional Comments

- The original raw dataset will remain unchanged.
- The geographical coverage and category definitions will be
  checked before cleaning.
- Missing values and duplicates will be investigated before
  deciding how to handle them.
- Economic inactivity and unemployment are different measures
  and will be kept distinct during preparation and analysis.

  ---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
if os.path.basename(os.getcwd()) == "jupyter_notebooks":
    os.chdir("..")

print(os.getcwd())

c:\Dudley employment barriers


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers'

## 1. Import Libraries

- os: manage the working directory and output folders.
- pandas: load, inspect, clean and save the dataset.

In [4]:
import os
import pandas as pd

---

## 2. Inspect the Raw Dataset

Check the first ten lines to identify column headings and any
introductory text before loading the CSV into pandas.

## Ethical Considerations When Inspecting Raw Socio‑Economic Data

Before loading the dataset into pandas, it is important to inspect the raw file structure.  
This step is not only technical — it carries **ethical responsibilities**, especially when working with sensitive socio‑economic indicators such as employment status, health limitations, transport access, and qualification levels across Dudley LSOAs.

### Why This Step Matters Ethically

Inspecting the raw dataset helps ensure:

- **Transparency** — understanding exactly what the dataset contains before transforming it.
- **Accuracy** — identifying metadata, units, dates, and contextual notes that affect interpretation.
- **Fairness** — preventing cleaning errors that could distort the representation of vulnerable communities.
- **Accountability** — documenting how the dataset is handled from the very first step.

The introductory lines in the file (title, copyright, units, date) confirm:
- The dataset originates from a trusted source (ONS/Nomis).
- The indicators refer to *real communities*, meaning misinterpretation can cause harm.

### Ethical Risks at the Raw Inspection Stage

If raw data is not inspected carefully, several ethical risks may arise:

- **Misaligned columns**  
  Could incorrectly inflate or reduce disadvantage in certain LSOAs.

- **Incorrect units or metadata**  
  May lead to misleading conclusions about employment barriers.

- **Malformed or missing rows**  
  Might disproportionately affect specific neighbourhoods, creating biased comparisons.

- **Ignoring contextual notes**  
  Could cause analysts to misinterpret what each indicator actually measures.

### Ethical Responsibility in Data Handling

By manually reviewing the raw file:

- You ensure that no structural issues introduce **accidental bias**.
- You protect communities from being misrepresented due to technical mistakes.
- You uphold ethical standards by treating socio‑economic data with care and respect.
- You demonstrate responsible practice in analysing sensitive public‑sector data.

### Summary

Inspecting raw data is an ethical safeguard.  
It ensures that the dataset is understood correctly, processed fairly, and used responsibly — especially when analysing employment barriers that affect real people in Dudley.



In [5]:
raw_path = "raw_datasets/dudley_economic_activity.csv"

with open(raw_path, encoding="utf-8-sig") as file:
    for _ in range(10):
        line = file.readline()
        if not line:
            break
        print(line.rstrip())


"TS066 - Economic activity status"
"ONS Crown Copyright Reserved [from Nomis on 14 October 2026]"
"Population :","All usual residents aged 16 years and over"
"Units      :","Persons"
"Date       :","2021"

"2021 super output area - lower layer","Total: All usual residents aged 16 years and over","Economically active (excluding full-time students)","Economically active (excluding full-time students):In employment","Economically active (excluding full-time students): Unemployed","Economically inactive: Student","Economically inactive: Looking after home or family","Economically inactive: Long-term sick or disabled","Economically inactive: Other"

"E01009763 : Dudley 001A",1637,970,888,82,97,164,73,70


### Findings from the Raw File Inspection

- The dataset is TS066: Economic activity status, from Census 2021.
- It covers all usual residents aged 16 years and over.
- Values represent counts of persons.
- The geography is 2021 Lower Layer Super Output Areas (LSOAs).
- The first six lines contain metadata and a blank line.
  These will be skipped when loading the dataset.
- The column headings appear on the seventh line.
- The first visible data record is Dudley 001A.

### 2.1 Load the Dataset

Load the CSV into a pandas DataFrame, skipping the six introductory
lines so the column headings are read correctly.

Display the dataset dimensions and the first and last five rows
to check the structure and identify any footer notes.

In [6]:
# Load the CSV, skipping the six introductory lines.
df = pd.read_csv(
    raw_path,
    skiprows=6,
    encoding="utf-8-sig"
)

# Show the dataset dimensions.
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

# Display the first five rows.
display(df.head())

Rows: 208, Columns: 9


,2021 super output area - lower layer,Total: All usual residents aged 16 years and over,Economically active (excluding full-time students),Economically active (excluding full-time students):In employment,Economically active (excluding full-time students): Unemployed,Economically inactive: Student,Economically inactive: Looking after home or family,Economically inactive: Long-term sick or disabled,Economically inactive: Other
0,E01009763 : Dudley 001A,1637,970.0,888.0,82.0,97.0,164.0,73.0,70.0
1,E01009764 : Dudley 001B,1264,759.0,731.0,28.0,35.0,50.0,35.0,31.0
2,E01009767 : Dudley 001C,1265,748.0,699.0,49.0,35.0,69.0,66.0,28.0
3,E01009769 : Dudley 001D,1431,701.0,618.0,83.0,78.0,119.0,148.0,59.0
4,E01009770 : Dudley 001E,1214,664.0,625.0,39.0,41.0,73.0,69.0,50.0


Section 2 content

### Findings from Loading and Inspecting the Full Dataset

- The imported dataset contains 208 rows and 9 columns.
- Rows 0–202 contain 203 Dudley LSOA records.
- Rows 203–206 contain disclosure-control notes.
- Row 207 is empty.
- Area codes and area names are combined in the first column.
- No missing values are visible within the 203 area records;
  this will be confirmed with a missing-value check.
- Count values are displayed with decimal places in several
  columns. Data types will be checked after removing non-data rows.
- The selected columns include an economically active subtotal
  and its employment and unemployment components. These must
  not be added together, as this would double-count people.
- The export does not include every economic activity category,
  so the selected categories should not be expected to sum
  to the total population.
- The source notes describe confidentiality adjustments to counts,
  which should be considered when interpreting small values.

The next step is to remove the five non-data rows while preserving
all 203 area records and keeping the original CSV unchanged.

---

## 3. Remove Non-Data Rows

Remove the four footer-note rows and the empty row by retaining
records with a valid Dudley LSOA code and name.

Create a separate working DataFrame, keeping the imported data
in df unchanged. Display all excluded rows and the row counts
to verify what was removed.

## Ethical Considerations When Removing Non‑Data Rows

This step filters the dataset to retain only valid Dudley LSOA records and removes footer notes, metadata lines, and empty rows. Although this is a technical cleaning operation, it carries important **ethical implications**, especially when working with socio‑economic indicators that describe real communities.

### Why Ethical Care Is Needed Here

The dataset includes:
- 203 genuine LSOA records  
- Several non‑data rows containing disclosure‑control notes  
- An empty row  

Removing these rows is necessary for accurate analysis, but it must be done **transparently and cautiously** to avoid unintentionally excluding valid data or misrepresenting communities.

### Ethical Risks in Filtering

If filtering is done incorrectly, several ethical risks arise:

- **Accidental removal of real LSOAs**  
  Over‑strict patterns or incorrect matching could delete genuine neighbourhoods, leading to biased results.

- **Inclusion of non‑data rows**  
  Keeping metadata or disclosure notes inside the dataset could distort counts, inflate totals, or mislead downstream analysis.

- **Misinterpretation of confidentiality adjustments**  
  The ONS applies small‑value adjustments to protect individuals.  
  Analysts must avoid treating these adjusted values as exact counts.

- **Loss of transparency**  
  If excluded rows are not shown, users cannot verify that the cleaning process was fair and accurate.

### Ethical Safeguards Used in This Step

This cleaning step includes several responsible practices:

- **Explicit pattern matching**  
  The regular expression only selects rows with a valid Dudley LSOA code and name, reducing the risk of accidental exclusion.

- **Displaying all removed rows**  
  Showing the excluded rows ensures transparency and allows manual verification.

- **Keeping the original DataFrame unchanged**  
  This preserves the raw data for auditing and prevents irreversible changes.

- **Reporting row counts before and after filtering**  
  This supports accountability and helps confirm that only non‑data rows were removed.

### Why This Matters for Fairness

Socio‑economic datasets often describe communities facing structural disadvantage.  
Cleaning decisions must therefore:

- Avoid introducing bias  
- Preserve all valid community records  
- Prevent misrepresentation of vulnerable groups  
- Maintain trustworthiness of the analysis

By carefully filtering and documenting each step, this project ensures that Dudley’s LSOAs are represented **fairly, accurately, and respectfully**.



In [7]:
# Developed with assistance from ChatGPT; output reviewed before proceeding.
# Identify records containing a Dudley LSOA code and name.
area_column = df.columns[0]
area_mask = df[area_column].str.fullmatch(
    r"E\d{8}\s*:\s*Dudley \d{3}[A-Z]",
    na=False
)

# Show every excluded row so the removal can be checked.
removed_rows = df.loc[~area_mask]
print("EXCLUDED ROWS:")
print(removed_rows.to_string(index=True))

# Create a working copy containing only Dudley area records.
df_clean = df.loc[area_mask].copy().reset_index(drop=True)

# Report the number of records before and after filtering.
print(f"\nOriginal rows: {len(df)}")
print(f"Removed rows: {len(removed_rows)}")
print(f"Remaining area records: {len(df_clean)}")
print(f"Columns: {df_clean.shape[1]}")

EXCLUDED ROWS:
    2021 super output area - lower layer                          Total: All usual residents aged 16 years and over  Economically active (excluding full-time students)  Economically active (excluding full-time students):In employment  Economically active (excluding full-time students): Unemployed  Economically inactive: Student  Economically inactive: Looking after home or family  Economically inactive: Long-term sick or disabled  Economically inactive: Other
203                                  NaN    In order to protect against disclosure of personal information, records                                                 NaN                                                               NaN                                                             NaN                             NaN                                                  NaN                                                NaN                           NaN
204                                  NaN  have been swapp

### Findings from Removing Non-Data Rows

- Removed four disclosure-control note rows and one empty row.
- Retained all 203 Dudley LSOA records and all 9 columns.
- The excluded rows contain no area records.
- The working DataFrame, df_clean, now contains only area data,
  with its index reset.
- The imported DataFrame, df, and the original CSV remain unchanged.
- The disclosure-control explanation remains documented above
  and preserved in the raw file.

---

## 4. Check Missing Values, Duplicates and Data Types

Check all 203 area records for missing values, fully duplicated
rows and repeated area identifiers.

Inspect the data type of every column before converting counts
to numeric types. Report any affected rows for investigation
before making further changes.

## Ethical Considerations When Checking Missing Values, Duplicates and Data Types

This step examines the cleaned dataset for missing values, duplicate records, and repeated area identifiers.  
Although this is a technical quality‑assurance process, it carries important **ethical responsibilities**, especially when analysing socio‑economic indicators for real Dudley communities.

### Why This Step Matters Ethically

Socio‑economic datasets describe structural barriers affecting people’s lives.  
Ensuring data quality is therefore essential for:

- **Fairness** — preventing inaccurate or biased comparisons between neighbourhoods.  
- **Accuracy** — ensuring indicators reflect real conditions without distortion.  
- **Transparency** — documenting how data integrity was verified.  
- **Non‑harm** — avoiding conclusions that misrepresent or stigmatise communities.

### Ethical Risks if Quality Checks Are Ignored

If missing values or duplicates are not identified and addressed, several ethical risks arise:

- **Missing values may cluster in specific LSOAs**  
  This could unfairly suggest lower population counts or reduced economic activity.

- **Duplicate rows may inflate disadvantage**  
  Repeated records could exaggerate unemployment or health‑related barriers.

- **Repeated area identifiers may distort comparisons**  
  If the same LSOA appears twice, it may appear to perform worse or better than reality.

- **Incorrect data types may misrepresent indicators**  
  For example, treating numeric counts as strings can break calculations and lead to misleading visualisations.

### Ethical Safeguards Used in This Step

This code implements several responsible practices:

- **Column‑level summaries**  
  Showing missing values and data types for every column supports transparency and accountability.

- **Full duplicate detection**  
  Checking both entire rows and area identifiers ensures no LSOA is accidentally over‑represented.

- **Displaying affected records**  
  Listing all problematic rows allows manual verification and prevents silent errors.

- **Clear reporting**  
  The output explicitly states whether any records require investigation, supporting ethical decision‑making.

### Why This Matters for Responsible Interpretation

Employment barriers, health limitations, and economic activity indicators are sensitive measures.  
Poor data quality can lead to:

- Incorrect policy conclusions  
- Misallocation of resources  
- Stigmatisation of neighbourhoods  
- Misleading dashboards used by non‑technical audiences

By performing and documenting these checks, this project ensures that Dudley’s LSOAs are represented **fairly, accurately, and respectfully**, and that subsequent analysis is grounded in reliable data.



In [8]:
# Developed with assistance from ChatGPT.

# Summarise missing values and data types for every column.
quality_summary = pd.DataFrame({
    "Data type": df_clean.dtypes.astype(str),
    "Missing values": df_clean.isna().sum()
})

print("COLUMN CHECKS:")
print(quality_summary.to_string())

# Identify all rows involved in duplicates.
duplicate_rows = df_clean.duplicated(keep=False)
duplicate_areas = df_clean.duplicated(
    subset=[area_column],
    keep=False
)

print(f"\nRows in duplicate groups: {duplicate_rows.sum()}")
print(f"Rows with repeated area identifiers: {duplicate_areas.sum()}")

# Display every record affected by missing values or duplicates.
affected_rows = (
    df_clean.isna().any(axis=1)
    | duplicate_rows
    | duplicate_areas
)

print("\nRECORDS REQUIRING INVESTIGATION:")
if affected_rows.any():
    print(df_clean.loc[affected_rows].to_string(index=True))
else:
    print("No missing values or duplicates found.")

COLUMN CHECKS:
                                                                 Data type  Missing values
2021 super output area - lower layer                                object               0
Total: All usual residents aged 16 years and over                   object               0
Economically active (excluding full-time students)                 float64               0
Economically active (excluding full-time students):In employment   float64               0
Economically active (excluding full-time students): Unemployed     float64               0
Economically inactive: Student                                     float64               0
Economically inactive: Looking after home or family                float64               0
Economically inactive: Long-term sick or disabled                  float64               0
Economically inactive: Other                                       float64               0

Rows in duplicate groups: 0
Rows with repeated area identifiers: 0

RECORD

### Findings from Data Quality Checks

- All 203 area records have no missing values across the 9 columns.
- No fully duplicated rows or repeated area identifiers were found.
- No records need to be removed or filled following these checks.
- The area identifier column has an object data type, suitable
  for its current text values.
- The total population column also has an object data type,
  because it originally included footer text.
- The remaining seven count columns have float64 data types.
- All eight count columns will be checked for numeric, whole-number
  and non-negative values before conversion to integer types.

---

## 5. Validate and Convert Count Columns

Check all eight count columns for non-numeric, missing, negative
or fractional values.

If every value passes these checks, convert the count columns
to integers because they represent numbers of people.
If any values fail, display the affected records for investigation
and leave the original columns unchanged.

## Ethical Considerations When Validating and Converting Count Columns

This step checks whether all eight count columns contain valid numeric values before converting them to integer types.  
Although this appears to be a purely technical operation, it carries important **ethical responsibilities**, especially when working with socio‑economic indicators that describe real Dudley communities.

### Why Ethical Validation Matters

The count columns represent numbers of people in each LSOA.  
Ensuring these values are valid is essential for:

- **Fairness** — preventing incorrect comparisons between neighbourhoods.  
- **Accuracy** — ensuring population counts and economic activity indicators are not distorted.  
- **Transparency** — documenting how data integrity was verified before transformation.  
- **Non‑harm** — avoiding misleading conclusions that could stigmatise communities.

### Ethical Risks if Validation Is Ignored

If invalid values are not detected, several ethical risks arise:

- **Non‑numeric values**  
  Could cause incorrect totals or misleading visualisations, affecting how communities are perceived.

- **Negative values**  
  Are impossible for population counts and may indicate data corruption that could unfairly distort results.

- **Fractional values**  
  Counts of people must be whole numbers. Fractional values may reflect confidentiality adjustments or export errors that require careful interpretation.

- **Silent conversion errors**  
  Automatically converting flawed data to integers could hide underlying issues and produce biased outputs.

### Ethical Safeguards Used in This Step

This code includes several responsible practices:

- **Coercing values to numeric with `errors="coerce"`**  
  Ensures that invalid entries are flagged rather than silently accepted.

- **Explicit checks for missing, negative, and fractional values**  
  Protects against misrepresentation of population counts.

- **Displaying affected records when issues are found**  
  Supports transparency and allows manual investigation before any transformation.

- **Conditional conversion**  
  Count columns are only converted to integers if *every* value passes validation, preventing accidental corruption.

- **Preserving the original data**  
  No changes are made to the raw dataset, maintaining auditability and accountability.

### Why This Matters for Responsible Interpretation

Employment barriers, economic activity, and population counts are sensitive indicators.  
Incorrect handling of these values can lead to:

- Misleading dashboards  
- Incorrect policy insights  
- Unfair comparisons between LSOAs  
- Stigmatisation of neighbourhoods  
- Misallocation of resources

By validating and converting count columns carefully, this project ensures that Dudley’s socio‑economic data is represented **fairly, accurately, and respectfully**, supporting ethical and responsible analysis.



In [9]:
# Developed with assistance from ChatGPT.

# Select the eight count columns, excluding the area identifier.
count_columns = df_clean.columns.drop(area_column)

# Create a numeric copy for validation.
numeric_counts = df_clean[count_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# Check every count for missing, non-numeric or invalid values.
invalid_counts = (
    numeric_counts.isna()
    | numeric_counts.lt(0)
    | numeric_counts.mod(1).ne(0)
)

print("INVALID VALUES PER COLUMN:")
print(invalid_counts.sum().to_string())

# Developed with assistance from ChatGPT.
# Convert only if all count values pass validation.
if invalid_counts.any().any():
    print("\nRECORDS REQUIRING INVESTIGATION:")
    print(
        df_clean.loc[invalid_counts.any(axis=1)].to_string(
            index=True
        )
    )
    print("\nNo column types were changed.")
else:
    df_clean = df_clean.astype({
        column: "int64" for column in count_columns
    })

    print("\nAll eight count columns converted to integers.")
    print("\nUPDATED DATA TYPES:")
    print(df_clean.dtypes.to_string())
    print(f"\nRows: {len(df_clean)}, Columns: {df_clean.shape[1]}")

INVALID VALUES PER COLUMN:
Total: All usual residents aged 16 years and over                   0
Economically active (excluding full-time students)                  0
Economically active (excluding full-time students):In employment    0
Economically active (excluding full-time students): Unemployed      0
Economically inactive: Student                                      0
Economically inactive: Looking after home or family                 0
Economically inactive: Long-term sick or disabled                   0
Economically inactive: Other                                        0

All eight count columns converted to integers.

UPDATED DATA TYPES:
2021 super output area - lower layer                                object
Total: All usual residents aged 16 years and over                    int64
Economically active (excluding full-time students)                   int64
Economically active (excluding full-time students):In employment     int64
Economically active (excluding full-time stu

### Findings from Count Validation and Conversion

- All eight count columns passed validation across all 203 records.
- No missing, non-numeric, negative or fractional values were found.
- All eight count columns were successfully converted to int64.
- This removes unnecessary decimal places while preserving
  the original count values.
- The area identifier column remains text.
- The dataset still contains 203 rows and 9 columns.

---

## 6. Separate Area Codes and Names

Split the combined area identifier into LSOA_Code and LSOA_Name.
The code will be used to join this dataset with other datasets,
once their geographical boundaries have been checked.

Keep the original identifier for now and display all 203 mappings
to verify the split.

## Ethical Considerations When Splitting and Validating Area Identifiers

This step separates the combined area identifier into two fields — `LSOA_Code` and `LSOA_Name` — and checks that all 203 records contain valid, unique, and complete identifiers.  
Although this is a structural data-cleaning operation, it carries important **ethical responsibilities**, because these identifiers represent real neighbourhoods in Dudley.

### Why Ethical Care Is Needed Here

LSOA codes and names are not just technical labels — they are the basis for:

- Joining datasets together  
- Mapping socio‑economic indicators  
- Comparing neighbourhoods  
- Communicating findings to non‑technical audiences  

Ensuring these identifiers are correct is essential for:

- **Fairness** — preventing misalignment between datasets that could misrepresent communities.  
- **Accuracy** — ensuring each LSOA is analysed correctly and not confused with another.  
- **Transparency** — documenting how geographic identifiers were handled.  
- **Non‑harm** — avoiding incorrect conclusions about specific neighbourhoods.

### Ethical Risks if Identifier Handling Is Incorrect

If area identifiers are split or validated incorrectly, several ethical risks arise:

- **Misassigned codes**  
  Could cause indicators from one neighbourhood to be attributed to another, leading to unfair comparisons.

- **Missing or malformed names**  
  May cause confusion or misinterpretation in dashboards or reports.

- **Duplicate codes**  
  Could inflate disadvantage or distort population counts for specific areas.

- **Silent errors during splitting**  
  If the delimiter is misinterpreted or whitespace is not removed, identifiers may become inconsistent.

### Ethical Safeguards Used in This Step

This code includes several responsible practices:

- **Splitting on the first colon only**  
  Ensures that names containing additional punctuation are not accidentally broken.

- **Whitespace removal**  
  Prevents subtle formatting inconsistencies that could cause join failures.

- **Displaying the first five mappings**  
  Supports transparency and allows manual verification of the split.

- **Checking uniqueness, missing values, and duplicates**  
  Ensures that every LSOA is represented fairly and accurately.

- **Preserving the original identifier column**  
  Maintains auditability and allows cross-checking with the raw dataset.

### Why This Matters for Responsible Interpretation

Geographic identifiers are the backbone of socio‑economic analysis.  
If they are mishandled, the consequences can include:

- Incorrect mapping of employment barriers  
- Misleading dashboards  
- Misallocation of resources  
- Stigmatisation of neighbourhoods  
- Loss of trust in the analysis

By carefully splitting and validating area identifiers, this project ensures that Dudley’s LSOAs are represented **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [12]:
# Developed with assistance from ChatGPT.

# Split on the first colon and remove surrounding whitespace.
area_parts = df_clean[area_column].str.split(
    ":", n=1, expand=True
)

df_clean["LSOA_Code"] = area_parts[0].str.strip()
df_clean["LSOA_Name"] = area_parts[1].str.strip()

# Display the first five identifier mappings.
print("AREA CODE AND NAME MAPPINGS:")
display(
    df_clean[
        [area_column, "LSOA_Code", "LSOA_Name"]
    ].head()
)

# Check the separated identifiers across all records.
print("\nIDENTIFIER CHECKS:")
print(f"Unique LSOA codes: {df_clean['LSOA_Code'].nunique()}")
print(f"Missing codes: {df_clean['LSOA_Code'].isna().sum()}")
print(f"Missing names: {df_clean['LSOA_Name'].isna().sum()}")
print(f"Duplicate codes: {df_clean['LSOA_Code'].duplicated().sum()}")
print(f"Rows: {len(df_clean)}, Columns: {df_clean.shape[1]}")

AREA CODE AND NAME MAPPINGS:


,2021 super output area - lower layer,LSOA_Code,LSOA_Name
0,E01009763 : Dudley 001A,E01009763,Dudley 001A
1,E01009764 : Dudley 001B,E01009764,Dudley 001B
2,E01009767 : Dudley 001C,E01009767,Dudley 001C
3,E01009769 : Dudley 001D,E01009769,Dudley 001D
4,E01009770 : Dudley 001E,E01009770,Dudley 001E



IDENTIFIER CHECKS:
Unique LSOA codes: 203
Missing codes: 0
Missing names: 0
Duplicate codes: 0
Rows: 203, Columns: 11


### Findings from Separating Area Codes and Names

- Successfully separated the area identifiers into LSOA_Code
  and LSOA_Name.
- The separated values match the original identifiers.
- All 203 LSOA codes are unique.
- No missing codes, missing names or duplicate codes were found.
- The original combined identifier has been retained for now.
- The dataset contains 203 rows and 11 columns after adding
  the two new fields.
- LSOA_Code provides a joining key for other datasets, subject
  to checking that their geographical boundaries are compatible.

---

## 7. Rename the columns

Shorten the column headings to make them easier to use in later
notebooks. Keep the age group and the exclusion of full-time students
clear in the relevant names.

Keep `LSOA_Code` and `LSOA_Name` consistent with the qualifications
dataset. Retain the original area identifier as `Original_Area`.

## Ethical Considerations When Renaming Column Headings

Renaming columns may appear to be a simple technical step, but it carries important **ethical responsibilities**, especially when working with socio‑economic indicators that describe real communities in Dudley.  
Clear, accurate, and transparent column names help prevent misinterpretation, support responsible analysis, and protect communities from being unfairly represented.

### Why Ethical Care Is Needed Here

The original column headings contain:
- Long descriptions  
- Embedded age‑group definitions  
- Notes about excluding full‑time students  
- Multiple economic activity categories  

These headings are meaningful and must be preserved accurately when shortened.  
Renaming them incorrectly could lead to:

- Misinterpretation of what each indicator measures  
- Incorrect comparisons between LSOAs  
- Misleading dashboards or reports  
- Stigmatisation of neighbourhoods based on misunderstood data  

### Ethical Risks if Renaming Is Done Incorrectly

If column names are shortened without care, several ethical risks arise:

- **Loss of important context**  
  For example, removing “excluding full‑time students” could distort employment statistics and unfairly represent younger populations.

- **Ambiguous or misleading names**  
  Vague labels may cause analysts or dashboard users to misunderstand what the data shows.

- **Inconsistent naming across datasets**  
  If LSOA_Code or LSOA_Name differ between datasets, joins may fail or misalign, causing incorrect results.

- **Reduced transparency**  
  Poorly renamed columns make it harder for others to understand or audit the analysis.

### Ethical Safeguards Used in This Step

This renaming process includes several responsible practices:

- **Preserving essential meaning**  
  Each new name keeps the original intent clear (e.g., “Economically_Active_Excl_FT_Students”).

- **Maintaining clarity for non‑technical users**  
  Shorter names improve readability in notebooks, dashboards, and reports.

- **Keeping geographic identifiers consistent**  
  `LSOA_Code` and `LSOA_Name` match the qualifications dataset, supporting fair and accurate dataset joining.

- **Retaining the original identifier**  
  The combined area identifier is preserved as `Original_Area` to maintain transparency and auditability.

- **Avoiding any change to underlying data values**  
  Only column labels are updated — the counts remain untouched.

### Why This Matters for Responsible Interpretation

Column names shape how people understand the dataset.  
Clear, accurate naming helps ensure:

- Fair comparisons between neighbourhoods  
- Accurate interpretation of employment barriers  
- Responsible communication in dashboards  
- Reduced risk of misrepresenting vulnerable communities  
- Greater trust in the analysis  

By renaming columns carefully and transparently, this project supports ethical, fair, and responsible socio‑economic analysis across Dudley’s LSOAs.



In [14]:
# Developed with assistance from ChatGPT.

# Match the original headings to shorter, descriptive names.
column_names = {
    "2021 super output area - lower layer": "Original_Area",
    "Total: All usual residents aged 16 years and over": (
        "Total_Aged_16_Plus"
    ),
    "Economically active (excluding full-time students)": (
        "Economically_Active_Excl_FT_Students"
    ),
    "Economically active (excluding full-time students):In employment": (
        "Employed_Excl_FT_Students"
    ),
    "Economically active (excluding full-time students): Unemployed": (
        "Unemployed_Excl_FT_Students"
    ),
    "Economically inactive: Student": "Inactive_Student",
    "Economically inactive: Looking after home or family": (
        "Inactive_Home_Or_Family"
    ),
    "Economically inactive: Long-term sick or disabled": (
        "Inactive_Long_Term_Sick_Or_Disabled"
    ),
    "Economically inactive: Other": "Inactive_Other"
}

# Apply the new headings without changing the data values.
# Reference:
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html
df_clean = df_clean.rename(columns=column_names)

# Display the headings to check the changes.
print("COLUMN NAMES:")
for column in df_clean.columns:
    print(column)

print(f"\nRows: {df_clean.shape[0]}, Columns: {df_clean.shape[1]}")

COLUMN NAMES:
Original_Area
Total_Aged_16_Plus
Economically_Active_Excl_FT_Students
Employed_Excl_FT_Students
Unemployed_Excl_FT_Students
Inactive_Student
Inactive_Home_Or_Family
Inactive_Long_Term_Sick_Or_Disabled
Inactive_Other
LSOA_Code
LSOA_Name

Rows: 203, Columns: 11


### Findings

The column headings now use shorter, descriptive names.
The relevant headings retain the exclusion of full-time students,
and `Total_Aged_16_Plus` identifies the population covered.

`LSOA_Code` and `LSOA_Name` match the names used in the qualifications
dataset. The original area identifier is retained as `Original_Area`.

The dataset still contains **203 rows and 11 columns**.
Only the headings changed; the data values remain unchanged.

---

## 8. Save the Cleaned Dataset

Save the current cleaned data as a separate CSV without the
DataFrame index. Keep the original column names and the separate
LSOA code and name columns.

Reload the saved file and compare it with df_clean to confirm
that the export preserves all values, columns and records.
The original raw CSV will remain unchanged.

## Ethical Considerations When Saving and Verifying the Cleaned Dataset

Saving a cleaned dataset may seem like a simple technical step, but it carries important **ethical responsibilities**, especially when working with socio‑economic indicators describing real communities in Dudley.  
This stage ensures that the cleaned data is preserved accurately, transparently, and without introducing new errors that could misrepresent neighbourhoods.

### Why Ethical Care Is Needed Here

Once data is cleaned, it becomes the foundation for:

- Visualisations  
- Dashboards  
- Statistical summaries  
- Cross‑dataset joins  
- Public‑facing insights  

If the saved dataset contains errors, missing values, or unintended changes, the consequences can cascade through the entire project and lead to **unfair or misleading conclusions**.

### Ethical Risks if Saving Is Done Incorrectly

If the dataset is saved or reloaded incorrectly, several ethical risks arise:

- **Loss of data integrity**  
  Incorrect exports may drop rows, alter values, or change column types.

- **Silent introduction of missing values**  
  Exporting or reloading may convert certain values to `NaN`, affecting population counts.

- **Duplicate records**  
  If duplicates appear during export, some LSOAs may be over‑represented.

- **Misalignment of geographic identifiers**  
  If `LSOA_Code` or `LSOA_Name` change during export, dataset joins may fail or misrepresent communities.

- **Reduced transparency**  
  Without verification, users cannot be confident that the cleaned dataset matches the intended version.

### Ethical Safeguards Used in This Step

This code implements several responsible practices:

- **Creating a dedicated output folder**  
  Keeps cleaned data separate from raw data, supporting transparency and auditability.

- **Saving without an index column**  
  Prevents accidental introduction of non‑data columns that could confuse downstream users.

- **Reloading the saved file**  
  Ensures that the exported dataset is identical to the in‑memory version.

- **Using `pd.testing.assert_frame_equal`**  
  Provides a strict, reliable check that every value, column, and row matches exactly.

- **Reporting missing values, duplicates, and identifier issues**  
  Confirms that the saved dataset remains clean, complete, and ethically reliable.

### Why This Matters for Responsible Interpretation

Socio‑economic datasets influence how communities are understood.  
Errors introduced during saving can lead to:

- Misleading dashboards  
- Incorrect policy insights  
- Unfair comparisons between neighbourhoods  
- Stigmatisation of vulnerable groups  
- Loss of trust in the analysis  

By saving and verifying the cleaned dataset carefully, this project ensures that Dudley’s LSOA‑level data is preserved **fairly, accurately, and respectfully**, supporting ethical and responsible analysis throughout the workflow.



In [15]:
# Developed with assistance from ChatGPT.

# Create the output folder if it does not already exist.
os.makedirs("cleaned_datasets", exist_ok=True)

# Save the cleaned data without adding an index column.
output_path = (
    "cleaned_datasets/dudley_economic_activity_cleaned.csv"
)
df_clean.to_csv(output_path, index=False)

# Reload the saved CSV and verify the complete dataset.
saved_df = pd.read_csv(output_path)
pd.testing.assert_frame_equal(saved_df, df_clean)

# Report the export and final quality checks.
print(f"Saved to: {output_path}")
print(f"Rows: {len(saved_df)}, Columns: {saved_df.shape[1]}")
print(f"Missing values: {saved_df.isna().sum().sum()}")
print(f"Duplicate rows: {saved_df.duplicated().sum()}")
print(f"Duplicate LSOA codes: {saved_df['LSOA_Code'].duplicated().sum()}")
print("Verification passed: saved data matches df_clean.")

Saved to: cleaned_datasets/dudley_economic_activity_cleaned.csv
Rows: 203, Columns: 11
Missing values: 0
Duplicate rows: 0
Duplicate LSOA codes: 0
Verification passed: saved data matches df_clean.


### Findings from Saving the Cleaned Dataset

- Saved the cleaned CSV to:
  `cleaned_datasets/dudley_economic_activity_cleaned.csv`
- The saved dataset contains 203 rows and 11 columns.
- No missing values, duplicate rows or duplicate LSOA codes remain.
- Reloading and comparing the saved file confirmed that it
  matches df_clean exactly.
- Original column names and the combined area identifier are
  retained alongside the separate LSOA_Code and LSOA_Name columns.
- The raw CSV remains unchanged.

Initial cleaning of the economic activity dataset is complete.
Further preparation will follow after cleaning the other datasets.

---

## Conclusions

The economic activity dataset has been cleaned and saved with
203 area records. Checks confirmed no missing values, duplicate
rows or duplicate LSOA codes. All eight count columns are integers.

The saved CSV was reloaded and verified against the cleaned
DataFrame. The original raw dataset remains unchanged.

## Ethical Considerations in the Final Conclusions

The cleaning and validation of the economic activity dataset is now complete, and the final checks confirm that the dataset is reliable, consistent, and ready for responsible analysis.  
Because this dataset describes real communities in Dudley, the conclusions must reflect not only technical success but also the **ethical responsibilities** upheld throughout the process.

### Ethical Significance of the Cleaning Outcome

The cleaned dataset now contains:

- 203 verified LSOA records  
- No missing values  
- No duplicate rows  
- No repeated LSOA codes  
- Eight count columns validated and converted to integers  
- Preserved geographic identifiers  
- A fully verified export that matches the cleaned DataFrame  

These results demonstrate that the dataset is structurally sound and ethically safe to use for further socio‑economic analysis.

### Ethical Risks Avoided Through Careful Cleaning

By performing thorough validation, the workflow avoided several potential harms:

- **Misrepresenting neighbourhoods** through missing or duplicated records  
- **Inflating or deflating population counts** due to incorrect data types  
- **Misaligned geographic identifiers** that could distort cross‑dataset joins  
- **Silent errors during export** that could mislead dashboards or reports  
- **Unfair comparisons** caused by inconsistent or corrupted values  

Avoiding these risks is essential when analysing employment barriers, health limitations, and economic activity — all of which relate to vulnerable groups.

### Ethical Strengths of the Completed Workflow

The cleaning process demonstrates:

- **Transparency** — every transformation was documented and justified  
- **Accuracy** — all values were validated before conversion  
- **Fairness** — no LSOA was removed, duplicated, or misrepresented  
- **Accountability** — the saved dataset was reloaded and verified  
- **Respect for communities** — the raw dataset remains unchanged and auditable  

These practices ensure that the dataset can be used responsibly in later notebooks, dashboards, and reports.

### Ethical Conclusion

The cleaned economic activity dataset is now ethically robust and technically reliable.  
It preserves the integrity of Dudley’s LSOA‑level information, supports fair and accurate analysis, and provides a trustworthy foundation for exploring employment barriers without misrepresenting or harming the communities described.

